In [1]:
# --- CONFIGURATION ---
URI = "bolt://localhost:7687"  # Direct local connection
AUTH = ("neo4j", "VzZ@BTLg@JBoG94NzPJg9VJeTHaSRyk2") 
DB_NAME = "neo4j"

In [28]:
# Example complete graph pipeline
import ollama
import time
import json
import re
from neo4j import GraphDatabase

# --- CONFIGURATION ---
EMBED_MODEL = "mxbai-embed-large"
LLM_MODEL = "qwen3-coder"

driver = GraphDatabase.driver(URI, auth=AUTH)

class GraphRAGPipeline:
    def __init__(self):
        self.driver = driver
        print("Initializing System Context...")
        self.cached_context = self.get_system_context()

    def log(self, stage, message, data=None):
        print(f"\n{'='*20} [DEBUG: {stage.upper()}] {'='*20}")
        print(message)
        if data:
            print(f"\n--- DATA ({stage}) ---")
            if isinstance(data, (dict, list)):
                print(json.dumps(data, indent=2, default=str))
            else:
                print(data)
        print("="*60)

    def get_system_context(self):
        """
        Retrieves the LIVE database state with grouped schema, 
        truncated property examples, and vector index samples.
        """
        with self.driver.session() as session:
            # 1. Grouped Graph Structure
            schema_query = """
            CALL apoc.meta.graph() YIELD nodes, relationships
            UNWIND relationships AS rel
            WITH startNode(rel).name AS source, type(rel) AS rel_type, endNode(rel).name AS target
            RETURN source, rel_type, target ORDER BY source
            """
            schema_data = session.run(schema_query).data()
            formatted_schema = [f"(:{r['source']})-[:{r['rel_type']}]->(:{r['target']})" for r in schema_data]
    
            # 2. Properties with Examples
            props_query = """
            CALL apoc.meta.nodeTypeProperties() 
            YIELD nodeLabels, propertyName, propertyTypes 
            WITH nodeLabels[0] AS label, propertyName, propertyTypes[0] AS type
            WHERE NOT propertyName IN ['embedding', 'embedding_summary']
            ORDER BY label, propertyName
            
            // Find a non-null sample for each specific property
            CALL (label, propertyName) {
                MATCH (n) 
                WHERE label IN labels(n) AND n[propertyName] IS NOT NULL
                RETURN n[propertyName] AS sample_value 
                LIMIT 1
            }
            
            RETURN label, propertyName, type, sample_value
            """
            props_data = session.run(props_query).data()
            
            props_dict = {}
            for p in props_data:
                label = p['label']
                if label not in props_dict:
                    props_dict[label] = []
                
                val = p['sample_value']
                # Truncate strings to 50 chars, otherwise stringify
                if isinstance(val, str):
                    example = f"'{val[:50]}...'" if len(val) > 50 else f"'{val}'"
                else:
                    example = str(val)
                    
                props_dict[label].append(f"\n  - {p['propertyName']} ({p['type']}): {example}")
    
            # 3. Vector Indexes with 'embedding_summary' samples
            index_query = """
            SHOW INDEXES YIELD name, type, labelsOrTypes 
            WHERE type = 'VECTOR' 
            RETURN name, labelsOrTypes[0] AS label
            """
            index_records = session.run(index_query).data()
            
            formatted_indexes = []
            for idx in index_records:
                label = idx['label']
                # Corrected: IS NOT NULL to find a real sample
                summary_sample = session.run(
                    f"MATCH (n:{label}) WHERE n.embedding_summary IS NOT NULL RETURN n.embedding_summary LIMIT 1"
                ).single()
                
                sample_text = summary_sample[0] if summary_sample else "No summary available"
                formatted_indexes.append(
                    f"Index: {idx['name']} (Label: {label})\n"
                    f"   -> Sample embedding_summary: \"{sample_text[:200]}...\""
                )
    
        # Building the Final Markdown String
        context = ["### LIVE DATABASE SCHEMA (LLM CONTEXT GUIDE)"]
        context.append("\n**1. GRAPH STRUCTURE (Grouped by Source):**")
        context.extend(list(dict.fromkeys(formatted_schema))) # Remove duplicates and keep order
        
        context.append("\n**2. NODE PROPERTIES & EXAMPLES:**")
        for label, properties in props_dict.items():
            context.append(f"- **{label}**: {'; '.join(properties)}")
            
        context.append("\n**3. VECTOR INDEXES (Use for semantic search):**")
        context.extend(formatted_indexes)
    
        return "\n".join(context)
        
    def get_embedding(self, text):
        res = ollama.embeddings(model=EMBED_MODEL, prompt=text)
        return res['embedding']

    def plan_execution(self, user_query, schema_context):
        """Step 1: Ask the AI to identify what needs embedding and the overall strategy."""
        prompt = f"""
        Role: You are the 'Strategic Planner' in a GraphRAG pipeline. Your task is to analyze the user question against the provided Neo4j schema to determine the execution strategy.

        ### 1. LIVE DATABASE CONTEXT:
        {schema_context}
        
        ### 2. USER QUESTION: 
        "{user_query}"
        
        ### 3. CLASSIFICATION TASK:
        Assign exactly one 'query_type' based on these definitions:
        - "stats": Requests for counts, lists, or simple aggregations existing within the schema.
        - "multi_step_analysis": Complex requests requiring path traversals, comparisons, or multi-node correlations.
        - "out_of_scope": Use this if the question cannot be answered using the provided Schema or Vector Indexes. You must carefully validate if the entities or relationships requested exist in the Context. If the data is missing, select this type.

        ### 4. ENTITY EXTRACTION:
        Identify all conceptual entities requiring semantic (vector) search. Map them to the correct 'embedding_name' found in the Vector Indexes section of the context.

        ### 5. OUTPUT REQUIREMENTS:
        - Provide a clear 'reasoning' for your classification. If 'out_of_scope', explicitly state what data is missing from the schema.
        - Return ONLY valid JSON.

        EXPECTED JSON FORMAT:
        {{
            "query_type": "stats" | "multi_step_analysis" | "out_of_scope",
            "reasoning": "Explain why this is stats vs analysis.",
            "embeddings_needed": [
                {{
                    "variable_name": "Unique variable name (e.g., emb_role)", 
                    "search_text": "Conceptual search term (e.g., 'Software Developer')", 
                    "embedding_name": "// Read the schema and carefully select the matching embedding index name"
                }},
                {{
                    "variable_name": "Next unique variable (e.g., emb_cert)", 
                    "search_text": "Next search term (e.g., 'Cloud Architecture')", 
                    "embedding_name": "// Select the matching embedding index name for this specific entity"
                }}
                // Add as many additional embedding objects as needed to satisfy the query.
            ]
        }}
        """
        self.log("Planning", "Analyzing intent and extracting entities...")
        
        res = ollama.generate(model=LLM_MODEL, prompt=prompt)
        
        try:
            # Clean potential markdown from LLM output
            clean_json = re.sub(r"```json|```", "", res['response']).strip()
            return json.loads(clean_json)
        except json.JSONDecodeError:
            self.log("Error", "Failed to parse JSON plan.", res['response'])
            return {"query_type": "fallback", "embeddings_needed": []}

    def generate_cypher_query(self, user_query, schema_context, plan):
        """Step 3: Generate Cypher using the dynamically created embedding variables."""
        query_type = plan.get("query_type")
        
        available_vars = "\n".join([
            f"- Parameter: ${item['variable_name']} | Target Embedding: '{item['embedding_name']}' | Search: '{item['search_text']}'" 
            for item in plan.get('embeddings_needed', [])
        ])

        if query_type == 'stats':
            prompt = f"""
        Role: Neo4j Cypher Expert. Translate the user query into a high-performance Cypher query using the Schema and Vector Parameters provided.

        ### 1. CONTEXT:
        SCHEMA: {schema_context}
        PARAMETERS: {available_vars}

        ### 2. STRICT SYNTAX RULES:
        - **Vector Search**: Use `CALL db.index.vector.queryNodes(target_embedding, 100000, $parameter_name) YIELD node, score`.
        - **Parameters**: Use the `$` prefix for all provided vector variables (e.g., $emb_role). Never hardcode strings in the vector call.
        - **Quality**: Apply `WHERE score > 0.8` unless the query implies a broader search.
        - **Efficiency**: Use only the necessary nodes and relationships. For simple counts, return `count(node)`.
        - **Regex**: Use `~ '(?i)...'` for case-insensitive string matching on non-vector properties.

        ### 3. REFERENCE EXAMPLE:
        Question: "Python experts at Google"
        Query: 
        CALL db.index.vector.queryNodes('experience_embeddings', 100000, $emb_role) YIELD node AS p, score 
        WHERE score > 0.8
        MATCH (p)-[:AT_COMPANY]->(c:Company) WHERE c.name =~ '(?i)google.*'
        RETURN p.name

        ### 4. TASK:
        User Question: "{user_query}"
        
        RETURN RAW CYPHER ONLY. NO MARKDOWN. NO EXPLANATION.
        """
        elif query_type == 'multi_step_analysis':
            prompt = f"""
            Role: Neo4j Graph Data Scientist & Strategic Researcher.
            
            ### PIPELINE CONTEXT:
            You are the 'Extraction Agent' in a 3-stage GraphRAG pipeline:
            1. Planner (Current Stage): You break the user's question into Cypher tasks.
            2. Executor: A Python runner executes these tasks against Neo4j.
            3. Synthesizer (Final Stage): A Final LLM reads the results of all tasks to answer the user.
            
            ### OBJECTIVE:
            Generate a sequence of Cypher queries that provide the Synthesizer with a clear, evidentiary data trail to answer: "{user_query}"

            ### SYSTEM CONSTRAINTS:
            1. **Live Schema**: {schema_context}
            2. **Parameters**: {available_vars}
            3. **Vector Syntax**: `CALL db.index.vector.queryNodes(target_embedding, top_k, $param) YIELD node, score WHERE score > 0.8`.
            4. **No Hallucinations**: Only use labels/properties provided in the schema context.
            
            ### STRICT VECTOR LIMIT & TRAVERSAL RULES:
            1. **Path-Walking/ID-Extraction**: You MUST set `top_k` to `1000`. 
            2. **Stats/Aggregation**: You MUST set `top_k` to `100000`. Never use arbitrary numbers like 50.

            ### CRITICAL CYPHER SYNTAX RULES:
            1. **ID Extraction**: ALWAYS use `elementId(node)`. NEVER use the deprecated `id(node)` function.
            2. **Single Column Return Rule**: Intermediate steps used to pass IDs to subsequent steps MUST return EXACTLY ONE column (`RETURN elementId(node) AS id`). Returning multiple columns will break the Python pipeline's list flattening logic.
            3. **Consuming Previous IDs**: Use `$step1_ids`, `$step2_ids`, etc., dynamically assigned by the executor. Example: `WHERE elementId(n) IN $step1_ids`. Do not invent parameter names.
            4. **Max Steps**: Generate a maximum of 3 steps.

            ### TASK DESIGN PHILOSOPHY:
            - **Discovery**: Steps 1 & 2 extract anchor IDs using embeddings with `top_k: 1000`.
            - **Correlation**: Middle steps bridge nodes via `MATCH` and `WHERE elementId(n) IN $stepN_ids`, returning a single column of target IDs.
            - **Synthesis**: The final step executes the comparison or counting using `UNWIND $stepN_results` to rehydrate and compare previous data.

            ### OUTPUT FORMAT:
            Return ONLY a JSON object with the following structure:
            {{
                "tasks": [
                    {{
                        "step": 1,
                        "description": "description of the cypher",
                        "cypher": "cypher here"
                    }}
                ]
            }}
            """
        else:
            return ""
        
        self.log("Cypher Generation", f"Generating {plan.get('query_type')} query using dynamic parameters...")
        # self.log("Cypher Generation", f"{prompt}")
        res = ollama.generate(model=LLM_MODEL, prompt=prompt)
        return re.sub(r"```json|```cypher|```", "", res['response']).strip()

    def execute_query(self, cypher, params):
        with self.driver.session() as session:
            try:
                result = session.run(cypher, **params)
                return result.data()
            except Exception as e:
                return [f"Cypher Error: {str(e)}"]

    def generate_final_answer(self, user_query, db_data):
        prompt = f"""
        Answer the User's Question based STRICTLY and ONLY on the 'Retrieved Data'.
        
        User Question: "{user_query}"
        Retrieved Data: {json.dumps(db_data, indent=2)}
        
        If data is empty or indicates an error, state that you could not find the information. Be direct and objective.
        """
        res = ollama.generate(model=LLM_MODEL, prompt=prompt)
        return res['response']
        
    def run(self, user_query):
        print(f"\n--- Processing: {user_query} ---")
        context = self.cached_context
        
        # 1. Plan Strategy
        plan = self.plan_execution(user_query, context)
        if plan.get('query_type') == 'out_of_scope':
            self.log("REJECTED", f"Reason: {plan.get('reasoning')}")
            return

        # 2. Generate Initial Embeddings
        query_params = {}
        for entity in plan.get('embeddings_needed', []):
            query_params[entity['variable_name']] = self.get_embedding(entity['search_text'])
            
        # 3. Handle Branching Logic
        raw_output = self.generate_cypher_query(user_query, context, plan)
        
        results_registry = {} # Stores results of each step for cross-referencing

        if plan.get('query_type') == 'multi_step_analysis':
            try:
                task_data = json.loads(raw_output)
                tasks = task_data.get("tasks", [])
                
                for task in tasks:
                    step_id = task['step']
                    cypher_query = task['cypher']
                    
                    self.log(f"Step {step_id}: {task['description']}", "Preparing execution...")
                    
                    for prev_step, prev_data in results_registry.items():
                        # Preserve original raw dict list
                        query_params[f"step{prev_step}_results"] = prev_data
                        
                        # Flatten the dictionaries into a scalar list for Neo4j 'IN' clauses
                        extracted_ids = []
                        if isinstance(prev_data, list) and len(prev_data) > 0 and isinstance(prev_data[0], dict):
                            extracted_ids = [list(record.values())[0] for record in prev_data]
                        
                        query_params[f"step{prev_step}_ids"] = extracted_ids

                    # Filter parameters to only log those actually used in this specific query
                    # Also mask large vector arrays to keep terminal output readable
                    used_params = {
                        k: ("<vector_data>" if "emb" in k else v) 
                        for k, v in query_params.items() if f"${k}" in cypher_query
                    }

                    # Log the exact Cypher statement and the parameters injected
                    print(f"Step {step_id} Cypher", cypher_query)

                    # Execute against Neo4j
                    step_result = self.execute_query(cypher_query, query_params)
                    
                    # Log the returned rows
                    print(f"Step {step_id} Result", f"Rows returned: {len(step_result)}")
                    
                    results_registry[step_id] = step_result
                
                final_data = results_registry
            except json.JSONDecodeError:
                self.log("Error", "Multi-step logic failed to return valid JSON tasks.")
                return
        else:
            # Handle standard 'stats' single-pass queries
            used_params = {k: ("<vector_data>" if "emb" in k else v) for k, v in query_params.items()}
            self.log("Stats Cypher Query", raw_output, used_params)
            final_data = self.execute_query(raw_output, query_params)
            self.log("Stats Result", f"Rows returned: {len(final_data)}", final_data)

        # 4. Final Synthesis
        answer = self.generate_final_answer(user_query, final_data)
        print(f"\n[FINAL ANSWER]: {answer}")

pipeline = GraphRAGPipeline()
pipeline.run("For UI/UX Designers, is the market shifting more towards 'Product Design' titles or 'Creative Direction'?")

Initializing System Context...

--- Processing: For UI/UX Designers, is the market shifting more towards 'Product Design' titles or 'Creative Direction'? ---

==================== [DEBUG: PLANNING] ====================
Analyzing intent and extracting entities...

==================== [DEBUG: CYPHER GENERATION] ====================
Generating multi_step_analysis query using dynamic parameters...

==================== [DEBUG: STEP 1: EXTRACT IDS OF EXPERIENCES RELATED TO 'UI/UX DESIGNER' ROLE USING VECTOR SEARCH WITH TOP_K=1000] ====================
Preparing execution...
Step 1 Cypher CALL db.index.vector.queryNodes('experience_embeddings', 1000, $emb_role_ui_ux_designer) YIELD node, score WHERE score > 0.8 RETURN elementId(node) AS id
Step 1 Result Rows returned: 1000

==================== [DEBUG: STEP 2: EXTRACT IDS OF EXPERIENCES RELATED TO 'PRODUCT DESIGN' ROLE USING VECTOR SEARCH WITH TOP_K=1000] ====================
Preparing execution...
Step 2 Cypher CALL db.index.vector.queryNo

In [66]:
def run_full_detail_recall(certification_search_text, role_filter_text="Developer", certification_threshold=0.9, role_threshold=0.9):
    # --- Step 1: Generating Dual Embeddings ---
    print(f"--- Step 1: Generating Dual Embeddings ---")
    certification_embedding_vector = ollama.embeddings(
        model="mxbai-embed-large", 
        prompt=certification_search_text
    )['embedding']
    
    role_identity_embedding_vector = ollama.embeddings(
        model="mxbai-embed-large", 
        prompt=role_filter_text
    )['embedding']

    with driver.session() as database_session:
        # --- STAGE A: Semantic Certification Search ---
        # Returns all certification nodes exceeding the similarity threshold
        certification_search_query = """
        CALL db.index.vector.queryNodes('certification_embeddings', 5000, $vector) 
        YIELD node AS certification_node, score AS similarity_score
        WHERE similarity_score > $threshold
        RETURN elementId(certification_node) AS certification_element_id, 
               certification_node.title AS certification_title, 
               similarity_score
        ORDER BY similarity_score DESC
        """
        
        certification_query_results = database_session.run(
            certification_search_query, 
            {"vector": certification_embedding_vector, "threshold": certification_threshold}
        ).data()
        
        # Mapping: {element_id: (title, score)}
        certification_metadata_lookup = {
            record['certification_element_id']: (record['certification_title'], record['similarity_score']) 
            for record in certification_query_results
        }
        target_certification_element_ids = list(certification_metadata_lookup.keys())

        print(f"\n--- Stage A: Matched Certifications (Score > {certification_threshold}) ---")
        for cert_id, (title, score) in certification_metadata_lookup.items():
            print(f" > {title} (Score: {score:.4f})")

        if not target_certification_element_ids:
            print("No certifications matched the threshold. Search terminated.")
            return

        # --- STAGE B: Graph Traversal to Professionals ---
        # Finds all individuals connected to the matched certifications
        professional_traversal_query = """
        MATCH (professional_node:Professional)-[:HAS_CERTIFICATION]-(certification_node:Certification)
        WHERE elementId(certification_node) IN $certification_ids
        RETURN elementId(professional_node) AS professional_element_id, 
               professional_node.name AS professional_name, 
               collect(elementId(certification_node)) AS held_certification_ids
        """
        
        traversal_results = database_session.run(
            professional_traversal_query, 
            {"certification_ids": target_certification_element_ids}
        ).data()
        
        target_professional_element_ids = [record['professional_element_id'] for record in traversal_results]
        professional_to_certification_map = {
            record['professional_element_id']: record['held_certification_ids'] 
            for record in traversal_results
        }

        print(f"\n--- Stage B: Graph Summary ---")
        print(f"Found {len(target_professional_element_ids)} unique professionals linked to certifications.")

        if not target_professional_element_ids:
            return

        # --- STAGE C: Semantic Experience/Role Verification ---
        # Checks work history against the 'Developer' concept via vector index
        experience_verification_query = """
        MATCH (professional_node:Professional)-[:HAS_EXPERIENCE]->(experience_node:Experience)
        WHERE elementId(professional_node) IN $professional_ids
        WITH professional_node, experience_node
        
        CALL db.index.vector.queryNodes('experience_embeddings', 10000, $vector) 
        YIELD node AS matched_experience_node, score AS identity_score
        WHERE matched_experience_node = experience_node AND identity_score > $threshold
        
        // Match the JobTitle node linked to this experience for extra detail
        OPTIONAL MATCH (experience_node)-[:ROLE_WAS]->(job_title_node:JobTitle)
        
        RETURN elementId(professional_node) AS professional_element_id, 
               professional_node.name AS professional_name, 
               max(identity_score) AS best_role_similarity_score,
               collect(DISTINCT job_title_node.name)[0] AS matched_job_title
        ORDER BY best_role_similarity_score DESC
        """
        
        final_verified_developers = database_session.run(
            experience_verification_query, {
                "professional_ids": target_professional_element_ids, 
                "vector": role_identity_embedding_vector, 
                "threshold": role_threshold
            }
        ).data()

        print(f"\n--- STAGE C: FINAL VERIFIED DEVELOPERS ---")
        print(f"Total verified candidates: {len(final_verified_developers)}")
        print("=" * 60)
        
        for developer_record in final_verified_developers[:10]:
            current_professional_id = developer_record['professional_element_id']
            
            print(f"DEVELOPER: {developer_record['professional_name']}")
            print(f" > Primary Match Title: {developer_record['matched_job_title'] or 'N/A'}")
            print(f" > Identity Similarity Score: {developer_record['best_role_similarity_score']:.4f}")
            print(f" > Relevant Certifications Found in Graph:")
            
            # Cross-reference the specific certs held by this person from our Stage A list
            matched_certification_ids = professional_to_certification_map.get(current_professional_id, [])
            for certification_id in matched_certification_ids:
                if certification_id in certification_metadata_lookup:
                    title, score = certification_metadata_lookup[certification_id]
                    print(f"   - {title} (Vector Match: {score:.4f})")
            print("-" * 40)

# Execution
run_full_detail_recall(
    certification_search_text="project management", 
    role_filter_text="Developer"
)

--- Step 1: Generating Dual Embeddings ---

--- Stage A: Matched Certifications (Score > 0.9) ---
No certifications matched the threshold. Search terminated.


In [64]:
def extract_transition_details(manual_qa_threshold=0.8, devops_threshold=0.8):
    print("--- Step 1: Generating Identity Vectors ---")
    manual_qa_identity_vector = ollama.embeddings(
        model="mxbai-embed-large", 
        prompt="QA engineer"
    )['embedding']
    
    # Loaded with highly specific tooling and concepts
    devops_identity_vector = ollama.embeddings(
        model="mxbai-embed-large", 
        prompt="DevOps engineer"
    )['embedding']

    with driver.session() as database_session:
        detail_extraction_query = """
        // 1. Anchor & Traverse
        CALL db.index.vector.queryNodes('experience_embeddings', 5000, $devops_vector) 
        YIELD node AS devops_exp, score AS devops_score
        WHERE devops_score > $devops_thresh
        
        MATCH (qa_exp:Experience)-[:FOLLOWED_BY]->(devops_exp)
        MATCH (p:Professional)-[:HAS_EXPERIENCE]->(qa_exp)
        
        // 2. Verify QA Identity & Temporal Filter
        WITH p, qa_exp, devops_exp, devops_score
        CALL db.index.vector.queryNodes('experience_embeddings', 10000, $qa_vector) 
        YIELD node AS matched_qa, score AS qa_score
        WHERE matched_qa = qa_exp AND qa_score > $qa_thresh AND devops_exp.duration_days < 730
        
        // 3. Fetch Rich Graph Details
        OPTIONAL MATCH (qa_exp)-[:ROLE_WAS]->(qa_title:JobTitle)
        OPTIONAL MATCH (qa_exp)-[:AT_COMPANY]->(qa_company:Company)
        
        OPTIONAL MATCH (devops_exp)-[:ROLE_WAS]->(devops_title:JobTitle)
        OPTIONAL MATCH (devops_exp)-[:AT_COMPANY]->(devops_company:Company)
        
        // 4. Return Comprehensive Row
        RETURN 
            p.name AS professional_name,
            p.linkedin_id AS id,
            
            qa_title.name AS qa_job_title,
            qa_company.name AS qa_company,
            qa_exp.start_date AS qa_start,
            qa_exp.end_date AS qa_end,
            coalesce(qa_exp.description, "No description provided") AS qa_desc,
            qa_score,
            
            devops_title.name AS devops_job_title,
            devops_company.name AS devops_company,
            devops_exp.start_date AS devops_start,
            devops_exp.end_date AS devops_end,
            coalesce(devops_exp.description, "No description provided") AS devops_desc,
            devops_score
            
        ORDER BY devops_score DESC
        """

        print("--- Step 2: Fetching Detailed Transition Graph ---")
        detailed_results = database_session.run(
            detail_extraction_query, 
            {
                "qa_vector": manual_qa_identity_vector, 
                "devops_vector": devops_identity_vector,
                "qa_thresh": manual_qa_threshold,
                "devops_thresh": devops_threshold
            }
        ).data()

        print(f"\n--- TRANSITION DETAILS ({len(detailed_results)} Profiles Found) ---")
        
        # Print out the first 10 profiles in a readable timeline format
        for index, record in enumerate(detailed_results[:10]):
            print(f"\n[{index + 1}] PROFESSIONAL: {record['id']}")
            print("-" * 50)
            
            # QA Phase
            qa_title = record['qa_job_title'] or "Unknown Title"
            qa_comp = record['qa_company'] or "Unknown Company"
            print(f"🗂️ PREVIOUS ROLE: {qa_title} @ {qa_comp} (Semantic Match: {record['qa_score']:.2f})")
            print(f"   Period: {record['qa_start']} to {record['qa_end']}")
            # Truncate description for terminal readability
            qa_desc_short = record['qa_desc'][:120].replace('\n', ' ') + ('...' if len(record['qa_desc']) > 120 else '')
            print(f"   Desc:   {qa_desc_short}")
            
            print("         |")
            print("         v (Transitioned to)")
            
            # DevOps Phase
            devops_title = record['devops_job_title'] or "Unknown Title"
            devops_comp = record['devops_company'] or "Unknown Company"
            print(f"🚀 NEW ROLE: {devops_title} @ {devops_comp} (Semantic Match: {record['devops_score']:.2f})")
            print(f"   Period: {record['devops_start']} to {record['devops_end']}")
            devops_desc_short = record['devops_desc'][:120].replace('\n', ' ') + ('...' if len(record['devops_desc']) > 120 else '')
            print(f"   Desc:   {devops_desc_short}")
            print("=" * 50)

# Run the extraction
extract_transition_details()

--- Step 1: Generating Identity Vectors ---
--- Step 2: Fetching Detailed Transition Graph ---

--- TRANSITION DETAILS (106 Profiles Found) ---

[1] PROFESSIONAL: trường-đặng-9999b1222
--------------------------------------------------
🗂️ PREVIOUS ROLE: mobile developer, quality control @ weallnet (Semantic Match: 0.81)
   Period: May 2021 to Jul 2022
   Desc:   No description provided
         |
         v (Transitioned to)
🚀 NEW ROLE: devops engineer @ dxc technology (Semantic Match: 0.88)
   Period: Aug 2022 to Sep 2022
   Desc:   No description provided

[2] PROFESSIONAL: linhcn1211
--------------------------------------------------
🗂️ PREVIOUS ROLE: devops engineer @ antsomi (Semantic Match: 0.80)
   Period: Apr 2024 to Apr 2025
   Desc:   No description provided
         |
         v (Transitioned to)
🚀 NEW ROLE: devops engineer @ vng corporation (Semantic Match: 0.87)
   Period: Jul 2025 to Sep 2025
   Desc:   No description provided

[3] PROFESSIONAL: phucnpt
------------------

In [4]:
def get_marketing_count():
    # 1. Generate Embedding for 'marketing'
    res = ollama.embeddings(model=EMBED_MODEL, prompt="marketing industry")
    marketing_vector = res['embedding']

    # 2. Execute Cypher Query
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    cypher = """
    CALL db.index.vector.queryNodes('professional_embeddings', 100000, $emb_industry) YIELD node, score
    WHERE score > 0.8
    MATCH (node)-[:CURRENT_INDUSTRY]->(:Industry)
    RETURN count(node) AS total_count
    """
    
    try:
        with driver.session() as session:
            result = session.run(cypher, emb_industry=marketing_vector)
            record = result.single()
            return record["total_count"] if record else 0
    finally:
        driver.close()

# --- EXECUTION ---
if __name__ == "__main__":
    count = get_marketing_count()
    print(f"Total nodes matching 'marketing': {count}")

Total nodes matching 'marketing': 12880
